# DS 340W Enhanced M-Help Experiment

**Project:** AI Detection of Mental Health Help-Seeking Signals from Social Media

**Goal:** classify a single Reddit post as either help-seeking (`help = 1`) or not help-seeking (`help = 0`).

This notebook is the enhanced experiment for the novelty/contribution assignment. The original baseline used word-level TF-IDF with logistic regression. Here, we compare multiple techniques and add a character n-gram TF-IDF model with Linear SVM as the main new approach.


## Important Scope

This project uses only individual Reddit posts. We do not use multiple posts per user, user-level modeling, or temporal/longitudinal modeling. MentalBART and other deep learning models are not implemented here; they are only future work.


In [2]:
import sys
print(sys.executable)

/Users/saisiddharth/Downloads/mhelp-project/venv/bin/python


In [3]:
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)


## Load Data

The original dataset has five columns: `ID`, `Text`, `help`, `Cause`, and `MH-condition`. For this project, we only use `Text` and `help`.


In [4]:
train = pd.read_csv(DATA_DIR / "train_data.csv")
val = pd.read_csv(DATA_DIR / "val_data.csv")
test = pd.read_csv(DATA_DIR / "test_data.csv")

print("Original columns:", train.columns.tolist())
print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)
train.head()


Original columns: ['ID', 'Text', 'help', 'Cause', 'MH-condition']
Train shape: (1142, 5)
Validation shape: (360, 5)
Test shape: (369, 5)


,ID,Text,help,Cause,MH-condition
0,ID-2,Will it actually get better ??\n\n\n\nI (26F) ...,1,love failure,"['BPD', 'GAD', 'MDD', 'SUD']"
1,ID-4,I can’t take it anymore\n\nI can’t take this s...,1,trauma and unresolved pain,"['BPD', 'MDD']"
2,ID-5,I struggle with self care in specific areas \n...,1,executive dysfunction,['GAD']
3,ID-6,Keep going\n\nYou may not know how or when you...,0,lack of support and mental health struggle,['MDD']
4,ID-7,Appointment for anxiety and depression - shoul...,1,ineffective treatment,"['BPD', 'GAD', 'MDD', 'SUD']"


## Keep Only Text and Binary Label


In [5]:
train = train[["Text", "help"]].dropna().copy()
val = val[["Text", "help"]].dropna().copy()
test = test[["Text", "help"]].dropna().copy()

train["help"] = train["help"].astype(int)
val["help"] = val["help"].astype(int)
test["help"] = test["help"].astype(int)

print("Train rows:", len(train))
print("Validation rows:", len(val))
print("Test rows:", len(test))


Train rows: 1142
Validation rows: 360
Test rows: 369


## Label Distribution

The labels are slightly imbalanced, with more help-seeking posts than non-help-seeking posts.


In [6]:
for split_name, split_df in [("Train", train), ("Validation", val), ("Test", test)]:
    print(split_name)
    print(split_df["help"].value_counts().sort_index())
    print()


Train
help
0    494
1    648
Name: count, dtype: int64

Validation
help
0    169
1    191
Name: count, dtype: int64

Test
help
0    159
1    210
Name: count, dtype: int64



## Text Cleaning

We use light text cleaning only. The goal is not to remove all informal writing because informal wording may be useful for detecting help-seeking signals.


In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_text"] = train["Text"].apply(clean_text)
val["clean_text"] = val["Text"].apply(clean_text)
test["clean_text"] = test["Text"].apply(clean_text)

train[["Text", "clean_text", "help"]].head(3)


,Text,clean_text,help
0,Will it actually get better ??\n\n\n\nI (26F) ...,will it actually get better ?? i (26f) have st...,1
1,I can’t take it anymore\n\nI can’t take this s...,i can’t take it anymore i can’t take this shit...,1
2,I struggle with self care in specific areas \n...,i struggle with self care in specific areas i ...,1


## Models Compared

We compare the original baseline against several enhanced techniques:

- Majority class baseline
- Original word TF-IDF + logistic regression
- Balanced logistic regression
- Word TF-IDF + Linear SVM
- Complement Naive Bayes
- **Novelty:** character n-gram TF-IDF + Linear SVM


In [8]:
models = {
    "Original: Word TF-IDF + Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Balanced Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=0.3,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Word TF-IDF + Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            sublinear_tf=True,
        )),
        ("clf", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Complement Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)),
        ("clf", ComplementNB()),
    ]),
    "Novelty: Character TF-IDF + Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char_wb",
            max_features=20000,
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True,
        )),
        ("clf", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}

list(models.keys())


['Original: Word TF-IDF + Logistic Regression',
 'Balanced Logistic Regression',
 'Word TF-IDF + Linear SVM',
 'Complement Naive Bayes',
 'Novelty: Character TF-IDF + Linear SVM']

## Evaluation Function

We report accuracy, macro F1, weighted F1, and help-class precision/recall/F1. Macro F1 is important because it gives equal weight to both classes.


In [9]:
def metric_row(model_name, split, y_true, y_pred):
    precision_help, recall_help, f1_help, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], zero_division=0
    )
    return {
        "Model": model_name,
        "Split": split,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted"),
        "Help Precision": precision_help[0],
        "Help Recall": recall_help[0],
        "Help F1": f1_help[0],
    }

def majority_predictions(y_train, n_rows):
    majority_label = int(y_train.value_counts().idxmax())
    return [majority_label] * n_rows


## Run All Models


In [10]:
rows = []
y_train = train["help"]

for split_name, split_df in [("Validation", val), ("Test", test)]:
    y_pred = majority_predictions(y_train, len(split_df))
    rows.append(metric_row("Majority Class Baseline", split_name, split_df["help"], y_pred))

for model_name, model in models.items():
    model.fit(train["clean_text"], train["help"])
    for split_name, split_df in [("Validation", val), ("Test", test)]:
        y_pred = model.predict(split_df["clean_text"])
        rows.append(metric_row(model_name, split_name, split_df["help"], y_pred))

results = pd.DataFrame(rows)
metric_cols = ["Accuracy", "Macro F1", "Weighted F1", "Help Precision", "Help Recall", "Help F1"]
results[metric_cols] = results[metric_cols].round(3)
results


,Model,Split,Accuracy,Macro F1,Weighted F1,Help Precision,Help Recall,Help F1
0,Majority Class Baseline,Validation,0.531,0.347,0.368,0.531,1.000,0.693
1,Majority Class Baseline,Test,0.569,0.363,0.413,0.569,1.000,0.725
2,Original: Word TF-IDF + Logistic Regression,Validation,0.594,0.561,0.568,0.584,0.822,0.683
3,Original: Word TF-IDF + Logistic Regression,Test,0.604,0.539,0.563,0.607,0.862,0.713
4,Balanced Logistic Regression,Validation,0.606,0.594,0.598,0.607,0.728,0.662
5,Balanced Logistic Regression,Test,0.591,0.562,0.578,0.617,0.743,0.674
6,Word TF-IDF + Linear SVM,Validation,0.594,0.577,0.582,0.593,0.754,0.664
7,Word TF-IDF + Linear SVM,Test,0.572,0.536,0.554,0.599,0.748,0.665
8,Complement Naive Bayes,Validation,0.556,0.412,0.430,0.545,0.990,0.703
9,Complement Naive Bayes,Test,0.583,0.407,0.451,0.578,0.990,0.730


## Save Comparison Table


In [11]:
results.to_csv(RESULTS_DIR / "comparison_results.csv", index=False)
print("Saved:", RESULTS_DIR / "comparison_results.csv")


Saved: ../results/comparison_results.csv


## Best Enhanced Model: Character TF-IDF + Linear SVM

This model gives the strongest test macro F1 and weighted F1 among our tested models. It is a better balanced model across both classes, even though the original logistic regression has slightly higher raw test accuracy.


In [12]:
best_model_name = "Novelty: Character TF-IDF + Linear SVM"
best_model = models[best_model_name]
best_model.fit(train["clean_text"], train["help"])
test_pred = best_model.predict(test["clean_text"])

print("Model:", best_model_name)
print(classification_report(test["help"], test_pred, target_names=["not help", "help"], zero_division=0))


Model: Novelty: Character TF-IDF + Linear SVM
              precision    recall  f1-score   support

    not help       0.55      0.45      0.49       159
        help       0.63      0.71      0.67       210

    accuracy                           0.60       369
   macro avg       0.59      0.58      0.58       369
weighted avg       0.60      0.60      0.60       369



## Confusion Matrix


In [13]:
cm = confusion_matrix(test["help"], test_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual not help", "Actual help"],
    columns=["Predicted not help", "Predicted help"],
)
cm_df.to_csv(RESULTS_DIR / "best_model_test_confusion_matrix.csv")
cm_df


,Predicted not help,Predicted help
Actual not help,72,87
Actual help,60,150


## Short Conclusion

The enhanced comparison shows why it is important to report more than one metric. The majority-class baseline and Complement Naive Bayes get high help-class recall because they predict the help class very often, but their macro F1 scores are weak. The character n-gram SVM gives the strongest test macro F1 and weighted F1 in this experiment, suggesting that character-level patterns can help detect informal help-seeking signals in Reddit posts.
